# Cert Prep — 03: Spark SQL

**Exam weight: ~15%**

Topics covered:
- spark.sql() — running SQL queries
- createOrReplaceTempView vs createOrReplaceGlobalTempView
- Window functions: rank, dense_rank, row_number, lag, lead
- Catalyst optimizer basics
- SQL functions: date, string, array, conditional
- Subqueries


In [ ]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.sql.warehouse.dir', '/workspace/data/training_warehouse')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

In [ ]:
# ── Temp Views ───────────────────────────────────────────────────────────────
print("=== Temp Views ===")

# Session-scoped — visible only in current SparkSession
df.createOrReplaceTempView("employees")

# Global — visible across SparkSessions in same JVM (rare in practice)
df.createOrReplaceGlobalTempView("employees_global")

# Query via SQL
result = spark.sql("""
    SELECT dept,
           COUNT(*)              AS headcount,
           ROUND(AVG(salary), 0) AS avg_salary,
           MAX(salary)           AS max_salary
    FROM employees
    WHERE salary IS NOT NULL
    GROUP BY dept
    ORDER BY avg_salary DESC
""")
result.show()

# Session view is gone when SparkSession ends
# Global view needs prefix: global_temp.employees_global
spark.sql("SELECT COUNT(*) FROM global_temp.employees_global").show()

In [ ]:
# ── Window Functions ─────────────────────────────────────────────────────────
print("=== Window Functions ===")
from pyspark.sql.window import Window

# Window spec: partition by dept, order by salary desc
w = Window.partitionBy("dept").orderBy(F.col("salary").desc_nulls_last())

df_win = df.withColumn("rank",        F.rank().over(w))            .withColumn("dense_rank",  F.dense_rank().over(w))            .withColumn("row_number",  F.row_number().over(w))            .withColumn("salary_lag",  F.lag("salary", 1).over(w))            .withColumn("salary_lead", F.lead("salary", 1).over(w))

df_win.select("name","dept","salary","rank","dense_rank","row_number",
              "salary_lag","salary_lead")       .orderBy("dept","rank").show()

print("rank()       — gaps after ties (1,1,3)")
print("dense_rank() — no gaps (1,1,2)")
print("row_number() — unique per row (1,2,3)")

In [ ]:
# ── Window: Aggregate Functions ──────────────────────────────────────────────
print("=== Window Aggregate Functions ===")
from pyspark.sql.window import Window

# Running total within dept
w_running = Window.partitionBy("dept").orderBy("salary")                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Dept-level stats without GROUP BY (keeps all rows)
w_dept = Window.partitionBy("dept")

df_agg = df.withColumn("dept_avg_salary", F.round(F.avg("salary").over(w_dept), 0))            .withColumn("dept_headcount",  F.count("*").over(w_dept))            .withColumn("running_sum",     F.sum("salary").over(w_running))            .withColumn("pct_of_dept",     F.round(F.col("salary") / F.sum("salary").over(w_dept) * 100, 1))

df_agg.select("name","dept","salary","dept_avg_salary","dept_headcount",
              "running_sum","pct_of_dept")       .orderBy("dept","salary").show()

In [ ]:
# ── Common SQL Functions ─────────────────────────────────────────────────────
print("=== Useful SQL / DataFrame Functions ===")

df2 = df.withColumn("hire_date_ts", F.to_date("hire_date", "yyyy-MM-dd"))

result = df2.select(
    "name",
    # String
    F.upper("name").alias("name_upper"),
    F.length("name").alias("name_len"),
    F.substring("name", 1, 3).alias("name_3"),
    # Date
    F.year("hire_date_ts").alias("hire_year"),
    F.datediff(F.current_date(), "hire_date_ts").alias("days_since_hire"),
    # Conditional
    F.when(F.col("salary") > 90000, "Senior")
     .when(F.col("salary") > 70000, "Mid")
     .otherwise("Junior").alias("level"),
    # Null handling
    F.coalesce("salary", F.lit(0.0)).alias("salary_safe"),
    F.isnull("salary").alias("salary_null"),
)
result.show()

In [ ]:
# ── Catalyst Optimizer ───────────────────────────────────────────────────────
print("=== Catalyst Optimizer ===")
print()
print("Catalyst is Spark SQL\'s query optimizer. It has 4 phases:")
print()
print("1. ANALYSIS    — resolve column names, check types, lookup catalog")
print("2. LOGICAL OPT — apply rules: predicate pushdown, constant folding,")
print("                 column pruning, filter pushdown")
print("3. PHYSICAL    — select join strategy, decide broadcast vs shuffle")
print("4. CODE GEN    — generate bytecode (Tungsten whole-stage codegen)")
print()

# Predicate pushdown — Spark pushes filter BELOW join (less data to shuffle)
q = (df.filter(F.col("dept") == "Engineering")
       .join(df.select("id","perf_score"), on="id")
       .filter(F.col("salary") > 90000))
print("Query plan (predicate pushdown visible):")
q.explain(mode="formatted")